<a href="https://colab.research.google.com/github/SiriusDarkz/riesgo-mora-cooperativa/blob/main/notebooks/caso1_cooperativa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Caso 1 · Riesgo de mora en préstamos nuevos
## Cooperativa Progreso del Sur

**Asignatura:** Selección y Validación de Modelos  
**Profesor:** Dr. Edian Franco  
**Equipo:** Jose Eugenio Duran Vizcaino · Anthony Burgos · Isaac Sanchez ·
Maximo Martinez · Francisco Jose Mejia

---

## El caso

La **Cooperativa Progreso del Sur** ofrece préstamos personales, comerciales
y para mejoras de vivienda a través de sucursales en Santo Domingo, San
Cristóbal, Baní y Azua. Durante el último año aumentaron las solicitudes
recibidas por canales digitales, pero el equipo de riesgo continúa revisando
manualmente buena parte de los expedientes.

La gerencia observa que algunas personas presentan atrasos importantes durante
los primeros meses del préstamo. Cuando esto ocurre, la cooperativa debe
realizar llamadas de cobro, renegociar condiciones y aumentar las provisiones
financieras. La gerente de riesgo plantea la necesidad de **identificar cuáles
solicitudes nuevas podrían presentar una mora superior a 30 días durante sus
primeros seis meses**.

Dos restricciones definen el problema:

- **Capacidad limitada:** el equipo de analistas solo puede revisar en
  detalle 120 solicitudes por semana.
- **Sin rechazo automático:** la gerencia no desea rechazar solicitantes con
  el modelo; desea priorizar cuáles expedientes necesitan verificación
  adicional.

## Qué construye este proyecto

> Un procedimiento que, cada semana, ordena las solicitudes nuevas por riesgo
> de mora y selecciona las 120 que el equipo de analistas debe revisar en
> detalle. La revisión funciona como tratamiento preventivo: verifica ingresos,
> pide garantías o ajusta condiciones, y así evita una parte de las moras que
> iban a ocurrir. El modelo no aprueba ni rechaza a nadie, solo apunta la
> capacidad limitada de revisión hacia donde más pérdida puede prevenir. Se
> recomendará implementarlo únicamente si demuestra, en un test honesto, que
> apunta mejor que la regla actual de la cooperativa.

Todo el ejercicio utiliza **datos sintéticos** generados en Python con semilla
fija, siguiendo las 8 etapas del protocolo del curso: formular, generar datos,
auditar variables, diseñar la evaluación, comparar contra baselines, congelar
el protocolo, evaluar en test una sola vez y recomendar.

# Etapa 1 · Formulación del problema

**Actor.** La gerencia de riesgo de la Cooperativa Progreso del Sur: la
gerente de riesgo define la política de revisión y su equipo de analistas la
ejecuta.

**Decisión.** Cuáles solicitudes nuevas de préstamo se envían cada semana a
verificación adicional, dentro del límite operativo de 120 revisiones
semanales.

**Acción.** Revisión manual detallada del expediente, que puede derivar en
verificación de ingresos, solicitud de garantías, reducción del monto o
cambio del plazo. La predicción no rechaza solicitantes: prioriza cuáles
revisar.

**Unidad de análisis.** Una solicitud de préstamo (una fila = una solicitud).
No es el cliente: un mismo cliente puede presentar varias solicitudes, lo que
obliga a controlar que sus solicitudes no queden repartidas entre desarrollo
y test.

**Momento de predicción.** Al recibir la solicitud completa, antes de la
decisión de aprobación. Elegimos este momento (y no "antes del desembolso")
porque la revisión adicional sirve precisamente para informar las condiciones
de aprobación. Consecuencias: (a) la variable `approved` aún no existe al
predecir, por lo que no puede usarse como predictora; (b) solo las
solicitudes aprobadas y desembolsadas llegan a tener etiqueta observada — una
limitación (etiquetas selectivas) que declaramos en el informe final.

**Horizonte.** Los primeros 6 meses de vida del préstamo, contados desde el
desembolso. La etiqueta de un préstamo solo se conoce cuando esta ventana se
cierra. *Supuesto de madurez:* asumimos que el análisis se realiza en una
fecha en la que todos los préstamos simulados ya completaron su ventana de
6 meses; en producción, la cooperativa solo podría entrenar con solicitudes
desembolsadas al menos 6 meses atrás, y las más recientes aún no tendrían
etiqueta observada.

**Variable objetivo.** `default_30d`: vale 1 si el préstamo alcanza una mora
superior a 30 días en algún momento durante sus primeros 6 meses; 0 en caso
contrario. Se deriva del seguimiento de atrasos (`days_past_due_6m`). Nótese
que combina dos números con roles distintos: los 30 días definen la severidad
del atraso que cuenta como evento; los 6 meses definen la ventana de
observación.

**Capacidad operativa.** 120 solicitudes por semana pueden recibir revisión
detallada. Esto convierte el problema en uno de **priorización**: el
procedimiento ordena las solicitudes de cada semana por riesgo estimado y las
120 primeras se revisan.

**Costo de los errores.**
- *Falso negativo* (no revisar una solicitud que luego cae en mora): costo
  financiero directo — provisiones, gestión de cobranza, renegociación y
  posible pérdida de capital.
- *Falso positivo* (gastar una revisión en una solicitud que habría pagado
  bien): horas de analista y fricción para un buen socio. Con capacidad fija,
  cada falso positivo tiene además un costo de oportunidad: desplaza del top
  120 a una solicitud riesgosa.

**Supuestos de costo (ilustrativos, fijados antes de la evaluación).**
Para traducir los errores a magnitudes comparables adoptamos supuestos
redondos, coherentes con los datos sintéticos que generaremos:

| Concepto | Supuesto |
|---|---|
| Monto promedio del préstamo | RD\$150,000 |
| Pérdida esperada si hay mora >30d (provisiones, cobranza, pérdida) | ≈20% del monto → RD\$30,000 |
| Costo de una revisión manual (≈2 horas de analista) | RD\$1,000 |
| Efecto de la revisión sobre una solicitud riesgosa | reduce la probabilidad de mora ≈35% |

Implicación: un falso negativo cuesta ~30 veces más que un falso positivo;
por eso consideramos más costoso el falso negativo y la métrica principal
medirá cuántas moras reales se capturan dentro de la capacidad semanal.
Además, el beneficio esperado de revisar un caso realmente riesgoso
(0.35 × RD\$30,000 ≈ RD\$10,500) supera con holgura el costo de la revisión
(RD\$1,000), lo que valida usar la capacidad completa.

El 35% es un **parámetro de diseño de la simulación**: el enunciado exige que
la revisión pueda disminuir la mora observada; nosotros fijamos su magnitud
en un valor moderado y plausible, y el generador de datos de la Etapa 2 usará
este mismo parámetro. Como el tratamiento es idéntico para cualquier método
de selección, la comparación entre el modelo y la regla vigente no depende
del valor exacto: variaciones razonables (20%–50%) cambian las cifras en
pesos, no la conclusión. Estas cifras son supuestos del ejercicio; en una
implementación real se calibrarían con la contabilidad de la cooperativa y
las normas de provisión aplicables.

**Criterio de no implementación.** El procedimiento no se recomienda si, con
las mismas 120 revisiones semanales en el período de test, no captura más
moras futuras que la regla operativa vigente (priorizar solicitudes con
deuda/ingreso alta y atrasos previos). Tampoco si la mejora, traducida a
dinero con los supuestos de costo anteriores, resulta demasiado pequeña para
compensar el costo de construir, mantener y monitorear el modelo.